In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mnli"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['premise', 'hypothesis', 'label', 'idx'],
    num_rows: 392702
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation_matched": raw_datasets['validation_matched'],
    "validation_mismatched": raw_datasets['validation_mismatched'],
    "test_matched": raw_datasets['test_matched'],
    "test_mismatched": raw_datasets['test_mismatched']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['premise'], raw_train_dataset[5]['hypothesis']

("my walkman broke so i'm upset now i just have to turn the stereo up real loud",
 "I'm upset that my walkman broke and now I have to turn the stereo up really loud.")

In [11]:
raw_train_dataset[5]['label']

0

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'premise': Value('string'),
 'hypothesis': Value('string'),
 'label': ClassLabel(names=['entailment', 'neutral', 'contradiction']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 9847/9847 [00:00<00:00, 13312.89 examples/s]


DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9847
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["premise", "hypothesis", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

validation_matched = DataLoader(
    tokenized_datasets["validation_matched"], batch_size=32, collate_fn=data_collator
)

validation_mismatched = DataLoader(
    tokenized_datasets["validation_mismatched"], batch_size=32, collate_fn=data_collator
)

eval_m_dataloader = DataLoader(
    tokenized_datasets["test_matched"], batch_size=32, collate_fn=data_collator
)

eval_mm_dataloader = DataLoader(
    tokenized_datasets["test_mismatched"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 83]),
 'token_type_ids': torch.Size([32, 83]),
 'attention_mask': torch.Size([32, 83])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.1568, grad_fn=<NllLossBackward0>) torch.Size([32, 3])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, validation_matched, task_name, device)[0]

{'accuracy': 0.844727457972491}

In [29]:
eval_loop(model, validation_mismatched, task_name, device)[0]

{'accuracy': 0.8488608624898292}

# 5. Submission Preparation

In [30]:
from train_eval_func import generate_glue_submission

In [31]:
generate_glue_submission(model, eval_m_dataloader, task_name, output_dir='.', output_filename='MNLI-m.tsv', device=device)

Generating GLUE submission for MNLI
Task type: multiclass


Predicting: 100%|██████████| 307/307 [00:35<00:00,  8.76it/s]


✓ Submission file created: ./MNLI-m.tsv
  Total predictions: 9796
  Format: index, prediction

First 5 rows:
   index  prediction
0      0  entailment
1      1     neutral
2      2  entailment
3      3     neutral
4      4  entailment


'./MNLI-m.tsv'

In [32]:
generate_glue_submission(model, eval_mm_dataloader, 'mnli-mm', output_dir='.', output_filename='MNLI-mm.tsv', device=device)

Generating GLUE submission for MNLI-MM
Task type: multiclass


Predicting: 100%|██████████| 308/308 [00:44<00:00,  6.95it/s]


✓ Submission file created: ./MNLI-mm.tsv
  Total predictions: 9847
  Format: index, prediction

First 5 rows:
   index     prediction
0      0     entailment
1      1        neutral
2      2        neutral
3      3        neutral
4      4  contradiction


'./MNLI-mm.tsv'